# ElasticNet â€” L1+L2 Regularized (Pipeline B)

**Model**: ElasticNet via `sklearn.linear_model.ElasticNet`
**Target**: `gdpc1`
**Variables**: Cat3 (53 + COVID = 56)
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: YES / **HP tuning**: YES (alpha + l1_ratio via `ElasticNetCV`) / **n_lags**: 4


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, split_for_scaler, load_data, PREDICTIONS_DIR, TARGET, COVID)

SEED=42; np.random.seed(SEED)
TRAIN_START="1959-01-01"; TRAIN_END="2007-12-31"; VAL_END="2016-12-31"
TEST_START="2017-01-01"; TEST_END="2025-12-31"
N_LAGS=4; VINTAGES = {"m1": -2, "m2": -1, "m3": 0, "post1": 1}
print("EN â€” Cat3, n_lags={}".format(N_LAGS))


EN â€” Cat3, n_lags=4


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)

# Phase A: HP tuning with widened l1_ratio grid
tune_data = monthly[(monthly["date"]>=TRAIN_START)&(monthly["date"]<=VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how="any").reset_index(drop=True)
feature_cols = [c for c in tune_flat.columns if c != "date" and c != TARGET and any(c == f or c.startswith(f + "_") for f in features)]

X_tune = tune_flat[feature_cols].values
y_tune = tune_flat[TARGET].values

# Keep scaling inside each TimeSeriesSplit fold to avoid validation-fold leakage.
tscv = TimeSeriesSplit(n_splits=5)
en_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(l1_ratio=[0.1,0.25,0.4,0.5,0.6,0.75,0.9,1.0],
                            alphas=np.logspace(-6,0,100), cv=tscv, random_state=SEED,
                            max_iter=20000, n_jobs=-1)),
])
en_pipe.fit(X_tune, y_tune)
en_cv = en_pipe.named_steps["model"]
best_a, best_l1 = en_cv.alpha_, en_cv.l1_ratio_
print("Best alpha={:.4g}, l1_ratio={:.3f}".format(best_a, best_l1))


Best alpha=2.477e-05, l1_ratio=1.000


In [3]:
test_quarters = monthly[(monthly["date"]>=TEST_START)&(monthly["date"]<=TEST_END)&
                         monthly["date"].dt.month.isin([3,6,9,12])]["date"].tolist()
predictions = {v:[] for v in VINTAGES}; actuals_list=[]; failed=0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly["date"]==pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, fc, N_LAGS
        )
        if len(train_fl)<10: predictions[vn].append(np.nan); continue
        sc_cols = [c for c in feature_cols if c in scaled_cols or any(c.startswith(s) for s in scaled_cols)]
        us_cols = [c for c in feature_cols if c in unscaled_cols or any(c.startswith(u) for u in unscaled_cols)]
        try:
            sc = StandardScaler()
            X_tr = np.hstack([sc.fit_transform(train_fl[sc_cols].values), train_fl[us_cols].values]) if us_cols else sc.fit_transform(train_fl[sc_cols].values)
            m = ElasticNet(alpha=best_a, l1_ratio=best_l1, max_iter=20000, random_state=SEED)
            m.fit(X_tr, train_fl[TARGET].values)
            X_te = np.hstack([sc.transform(tr_fl[sc_cols].values), tr_fl[us_cols].values]) if us_cols else sc.transform(tr_fl[sc_cols].values)
            predictions[vn].append(m.predict(X_te)[0])
        except Exception:
            predictions[vn].append(np.nan); failed+=1
    if (i+1)%8==0: print("  {} / {}".format(i+1, len(test_quarters)))
print("Done. {} failures.".format(failed))


  8 / 36


  16 / 36


  24 / 36


  32 / 36


Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({"date":test_quarters,"actual":actuals_list,"prediction":predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR,"elasticnet_{}.csv".format(vn)), index=False)
    print("Saved elasticnet_{}.csv".format(vn))


Saved elasticnet_m1.csv
Saved elasticnet_m2.csv
Saved elasticnet_m3.csv
Saved elasticnet_post1.csv


In [5]:
def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan
def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan
panels={"Pre-COVID":"2017-01-01,2019-12-31","COVID":"2020-04-01,2021-12-31",
        "Post-COVID":"2022-01-01,2025-12-31","Full":"2017-01-01,2025-12-31"}
a=np.array(actuals_list); d=pd.to_datetime(test_quarters)
print("EN alpha={:.4g}, l1_ratio={:.3f}".format(best_a, best_l1))
for pn, rng in panels.items():
    ps,pe=rng.split(","); m=(d>=ps)&(d<=pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp=np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}".format(vn,rmse(a[m],pp[m]),mae(a[m],pp[m])))


EN alpha=2.477e-05, l1_ratio=1.000
--- Pre-COVID ---
  m1  RMSFE=0.002713  MAE=0.002137
  m2  RMSFE=0.002732  MAE=0.002063
  m3  RMSFE=0.002703  MAE=0.002001
  post1  RMSFE=0.002702  MAE=0.001991
--- COVID ---
  m1  RMSFE=0.043706  MAE=0.027414
  m2  RMSFE=0.041985  MAE=0.026353
  m3  RMSFE=0.041379  MAE=0.025921
  post1  RMSFE=0.041081  MAE=0.025855
--- Post-COVID ---
  m1  RMSFE=0.004486  MAE=0.003363
  m2  RMSFE=0.004492  MAE=0.003420
  m3  RMSFE=0.004523  MAE=0.003451
  post1  RMSFE=0.004443  MAE=0.003438
--- Full ---
  m1  RMSFE=0.019869  MAE=0.008114
  m2  RMSFE=0.019133  MAE=0.007906
  m3  RMSFE=0.018868  MAE=0.007807
  post1  RMSFE=0.018759  MAE=0.007809
